# 🧴 PurePick Skin Analysis Model — Training Notebook

**Model:** EfficientNet-B4 (multi-label skin condition classifier)  
**Dataset:** DermNet (Kaggle) — 23 skin condition categories  
**Output:** `purepick_skin_model.pt` — drop into `backend/skin_analysis/ml_models/`

## Setup Instructions
1. Runtime → Change runtime type → **T4 GPU**
2. Add your Kaggle API key in the cell below
3. Run All Cells (Runtime → Run all)
4. Download `purepick_skin_model.pt` + `purepick_skin_classes.json` when training completes

**Estimated training time:** ~40 minutes on T4 GPU

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠ No GPU detected — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q timm kaggle albumentations scikit-learn matplotlib seaborn tqdm
print('✅ Dependencies installed')

In [ ]:
# ── Cell 3: Kaggle API setup ──────────────────────────────────────────────────
# Get your key from: https://www.kaggle.com/settings → API → Create New Token
# Upload the downloaded kaggle.json here OR paste credentials below

import os, json

# OPTION A: Upload kaggle.json (recommended)
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials configured')

In [ ]:
# ── Cell 4: Download DermNet dataset ─────────────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/shubhamgoel27/dermnet
# Same dataset used in the original skin analysis project

!kaggle datasets download -d shubhamgoel27/dermnet --quiet
!unzip -q dermnet.zip -d /content/dermnet
print('✅ DermNet dataset downloaded')

import os
base = '/content/dermnet'
for d in os.listdir(base):
    path = os.path.join(base, d)
    if os.path.isdir(path):
        count = sum(len(files) for _, _, files in os.walk(path))
        print(f'  {d}: {count} images')

In [ ]:
# ── Cell 5: Map DermNet categories → PurePick conditions ─────────────────────
# DermNet has 23 categories. We map them to our 14 clinical conditions.
# Multi-label: one image can belong to multiple PurePick conditions.

import os, shutil, random
from pathlib import Path

# ── PurePick condition labels ─────────────────────────────────────────────────
PUREPICK_CONDITIONS = [
    'Acne',
    'Dark_Circles',
    'Hyperpigmentation',
    'Rosacea_Redness',
    'Eczema_Dermatitis',
    'Dry_Skin',
    'Oily_Skin',
    'Fine_Lines_Wrinkles',
    'Enlarged_Pores',
    'Acne_Scars',
    'Melasma_Dark_Spots',
    'Psoriasis',
    'Normal_Skin',
    'Photodamage',
]

NUM_CLASSES = len(PUREPICK_CONDITIONS)
print(f'Training {NUM_CLASSES} condition classes:')
for i, c in enumerate(PUREPICK_CONDITIONS):
    print(f'  [{i:02d}] {c}')

# ── DermNet folder → PurePick condition mapping ───────────────────────────────
# Map each DermNet category to one or more PurePick conditions
DERMNET_TO_PUREPICK = {
    # Acne
    'Acne and Rosacea Photos':          ['Acne', 'Rosacea_Redness'],
    'Acne Keloidalis Nuchae Photos':    ['Acne', 'Acne_Scars'],
    # Pigmentation
    'Melanocytic Nevi & Melanoma':      ['Hyperpigmentation', 'Photodamage'],
    'Tinea Ringworm Candidiasis and other Fungal Infections': ['Hyperpigmentation'],
    # Redness / Rosacea
    'Seborrheic Keratoses and other Benign Tumors': ['Rosacea_Redness', 'Photodamage'],
    # Eczema / Dermatitis
    'Atopic Dermatitis Photos':         ['Eczema_Dermatitis', 'Dry_Skin'],
    'Exanthems and Drug Eruptions':     ['Eczema_Dermatitis'],
    'Eczema Photos':                    ['Eczema_Dermatitis', 'Dry_Skin'],
    # Dry / Texture
    'Psoriasis pictures Lichen Planus and related diseases': ['Psoriasis', 'Dry_Skin'],
    'Urticaria Hives':                  ['Dry_Skin', 'Eczema_Dermatitis'],
    # Scars
    'Nail Fungus and other Nail Disease': ['Acne_Scars'],
    'Warts Molluscum and other Viral Infections': ['Acne_Scars'],
    # Photodamage / Aging
    'Light Diseases and Disorders of Pigmentation': ['Photodamage', 'Melasma_Dark_Spots', 'Hyperpigmentation'],
    'Lupus and other Connective Tissue diseases': ['Rosacea_Redness', 'Photodamage'],
    # Vascular / Redness
    'Vascular Tumors':                  ['Rosacea_Redness'],
    'Vasculitis Photos':                ['Rosacea_Redness'],
    # Normal / other
    'Hair Loss Photos Alopecia and other Hair Diseases': ['Normal_Skin'],
    'Scabies Lyme Disease and other Infestations and Bites': ['Eczema_Dermatitis'],
    'Poison Ivy Photos and other Contact Dermatitis': ['Eczema_Dermatitis'],
    'Bullous Disease Photos':           ['Eczema_Dermatitis'],
    'Systemic Disease':                 ['Rosacea_Redness'],
    'Cellulitis Impetigo and other Bacterial Infections': ['Acne', 'Rosacea_Redness'],
}

print('\n✅ Condition mapping defined')

In [ ]:
# ── Cell 6: Build dataset with multi-label CSV ────────────────────────────────

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

records = []
dermnet_root = '/content/dermnet'

# Find the actual train folder
for sub in ['train', 'test', '']:
    candidate = os.path.join(dermnet_root, sub)
    if os.path.isdir(candidate):
        cats = [d for d in os.listdir(candidate) if os.path.isdir(os.path.join(candidate, d))]
        if len(cats) > 5:
            dermnet_root = candidate
            break

print(f'Dataset root: {dermnet_root}')
print(f'Categories found: {os.listdir(dermnet_root)[:5]}...')

for dermnet_cat, purepick_labels in DERMNET_TO_PUREPICK.items():
    cat_path = os.path.join(dermnet_root, dermnet_cat)
    if not os.path.isdir(cat_path):
        # Try case-insensitive match
        for d in os.listdir(dermnet_root):
            if d.lower() == dermnet_cat.lower():
                cat_path = os.path.join(dermnet_root, d)
                break
    if not os.path.isdir(cat_path):
        continue

    for fname in os.listdir(cat_path):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        img_path = os.path.join(cat_path, fname)
        label_vec = [1 if c in purepick_labels else 0 for c in PUREPICK_CONDITIONS]
        records.append({'image_path': img_path, **dict(zip(PUREPICK_CONDITIONS, label_vec))})

df = pd.DataFrame(records)
print(f'\nTotal images: {len(df)}')
print('\nCondition distribution:')
for c in PUREPICK_CONDITIONS:
    if c in df.columns:
        print(f'  {c}: {df[c].sum()} images')

# Train/val split (85/15)
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
print(f'\nTrain: {len(train_df)} | Val: {len(val_df)}')

train_df.to_csv('/content/train.csv', index=False)
val_df.to_csv('/content/val.csv', index=False)
print('✅ Dataset CSV files saved')

In [ ]:
# ── Cell 7: Dataset class + augmentation pipeline ─────────────────────────────

import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGE_SIZE = 380   # EfficientNet-B4 native resolution

# ── Training augmentation — aggressive for skin images ───────────────────────
TRAIN_TRANSFORM = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),

    # Spatial augmentations
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.4),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),

    # Colour augmentations — critical for skin tone invariance
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.4),

    # Simulate lighting / camera conditions
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.GaussNoise(var_limit=(5, 30), p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),

    # Randomly apply CLAHE during training (teaches model to handle contrast variations)
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),

    # Cutout (helps model not overfit to single region)
    A.CoarseDropout(max_holes=4, max_height=30, max_width=30, p=0.2),

    # Normalise to ImageNet stats (EfficientNet pretrained on ImageNet)
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

VAL_TRANSFORM = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])


class SkinDataset(Dataset):
    def __init__(self, df: pd.DataFrame, condition_cols: list, transform=None):
        self.df           = df.reset_index(drop=True)
        self.condition_cols = condition_cols
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_path  = row['image_path']

        img = cv2.imread(img_path)
        if img is None:
            # Return a blank image if file is corrupt
            img = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(image=img)['image']

        labels = torch.tensor(
            [row[c] for c in self.condition_cols], dtype=torch.float32
        )
        return img, labels


# Build loaders
train_df = pd.read_csv('/content/train.csv')
val_df   = pd.read_csv('/content/val.csv')

train_ds = SkinDataset(train_df, PUREPICK_CONDITIONS, TRAIN_TRANSFORM)
val_ds   = SkinDataset(val_df,   PUREPICK_CONDITIONS, VAL_TRANSFORM)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print('✅ DataLoaders ready')

In [ ]:
# ── Cell 8: Build EfficientNet-B4 model ──────────────────────────────────────

import timm
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')


class PurePickSkinModel(nn.Module):
    """
    EfficientNet-B4 backbone with custom multi-label classification head.

    Architecture:
      EfficientNet-B4 (pretrained ImageNet)
        → Global Average Pooling (built-in)
        → Dropout(0.4)            (prevents overfitting on small dataset)
        → Linear(1792, 512)
        → BatchNorm + ReLU
        → Dropout(0.3)
        → Linear(512, NUM_CLASSES)
        → Sigmoid              (multi-label: independent probability per condition)
    """
    def __init__(self, num_classes: int, pretrained: bool = True):
        super().__init__()
        # Load EfficientNet-B4 pretrained on ImageNet
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=pretrained,
            num_classes=0,          # Remove original classifier
            global_pool='avg',
        )
        in_features = self.backbone.num_features   # 1792 for B4

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        logits   = self.classifier(features)
        return logits  # raw logits — sigmoid applied in loss

    def predict_proba(self, x):
        """Inference: returns per-class probabilities 0-1."""
        return torch.sigmoid(self.forward(x))


model = PurePickSkinModel(num_classes=NUM_CLASSES).to(DEVICE)
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print('✅ EfficientNet-B4 model created')

In [ ]:
# ── Cell 9: Loss function + optimizer + scheduler ─────────────────────────────

import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR

EPOCHS    = 25
LR        = 3e-4
THRESHOLD = 0.40   # probability threshold for positive prediction

# ── Loss: Binary Cross Entropy with Logits ────────────────────────────────────
# Handles class imbalance via pos_weight: rarer conditions get higher weight
label_counts  = train_df[PUREPICK_CONDITIONS].sum(axis=0)
total_samples = len(train_df)
pos_weights   = torch.tensor(
    [(total_samples - c) / (c + 1e-6) for c in label_counts],
    dtype=torch.float32
).clamp(1.0, 10.0).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# ── Optimizer: AdamW with weight decay ───────────────────────────────────────
# Two param groups: lower LR for pretrained backbone, higher for new head
optimizer = optim.AdamW([
    {'params': model.backbone.parameters(),   'lr': LR * 0.1},
    {'params': model.classifier.parameters(), 'lr': LR},
], weight_decay=1e-4)

# ── Scheduler: OneCycleLR — best for transfer learning ───────────────────────
scheduler = OneCycleLR(
    optimizer,
    max_lr=[LR * 0.1, LR],
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1,
    anneal_strategy='cos',
)

print(f'Positive weights: {pos_weights.cpu().numpy().round(2)}')
print('✅ Training configuration ready')

In [ ]:
# ── Cell 10: Training loop ────────────────────────────────────────────────────

import time
from sklearn.metrics import f1_score, roc_auc_score


def compute_metrics(all_labels, all_preds, all_probs):
    """Compute F1 (macro) and ROC-AUC for multi-label classification."""
    try:
        f1    = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        # ROC-AUC only if both classes present in labels
        auc_scores = []
        for i in range(all_labels.shape[1]):
            if len(set(all_labels[:, i])) > 1:
                auc_scores.append(roc_auc_score(all_labels[:, i], all_probs[:, i]))
        auc = np.mean(auc_scores) if auc_scores else 0.0
        return f1, auc
    except Exception:
        return 0.0, 0.0


history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': []}
best_val_f1  = 0.0
best_model_path = '/content/purepick_skin_model_best.pt'

print(f'Starting training: {EPOCHS} epochs on {DEVICE}')
print('=' * 70)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    val_loss   = 0.0
    all_labels = []
    all_preds  = []
    all_probs  = []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            val_loss += loss.item()
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs >= THRESHOLD).astype(int)
            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    val_loss   /= len(val_loader)
    all_labels  = np.vstack(all_labels)
    all_preds   = np.vstack(all_preds)
    all_probs   = np.vstack(all_probs)
    val_f1, val_auc = compute_metrics(all_labels, all_preds, all_probs)

    elapsed = time.time() - t0
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['val_auc'].append(val_auc)

    marker = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        marker = ' ⭐ BEST'

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'F1: {val_f1:.4f} | '
          f'AUC: {val_auc:.4f} | '
          f'{elapsed:.0f}s{marker}')

print('\n' + '='*70)
print(f'Training complete. Best Val F1: {best_val_f1:.4f}')

In [ ]:
# ── Cell 11: Per-condition accuracy report ────────────────────────────────────

from sklearn.metrics import classification_report

# Load best checkpoint
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_labels, all_preds, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        logits = model(imgs.to(DEVICE))
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs >= THRESHOLD).astype(int)
        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels.numpy())

all_labels = np.vstack(all_labels)
all_preds  = np.vstack(all_preds)
all_probs  = np.vstack(all_probs)

print('Per-condition accuracy report:')
print('=' * 70)
for i, cond in enumerate(PUREPICK_CONDITIONS):
    tp = int(((all_preds[:, i] == 1) & (all_labels[:, i] == 1)).sum())
    fp = int(((all_preds[:, i] == 1) & (all_labels[:, i] == 0)).sum())
    fn = int(((all_preds[:, i] == 0) & (all_labels[:, i] == 1)).sum())
    prec = tp / (tp + fp + 1e-6)
    rec  = tp / (tp + fn + 1e-6)
    f1   = 2 * prec * rec / (prec + rec + 1e-6)
    try:
        auc = roc_auc_score(all_labels[:, i], all_probs[:, i]) if len(set(all_labels[:, i])) > 1 else 0.0
    except Exception:
        auc = 0.0
    print(f'  {cond:<30} F1={f1:.3f}  AUC={auc:.3f}  (TP={tp} FP={fp} FN={fn})')

In [ ]:
# ── Cell 12: Plot training curves ────────────────────────────────────────────

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(epochs_range, history['val_f1'])
axes[1].set_title('Validation F1 Score')

axes[2].plot(epochs_range, history['val_auc'])
axes[2].set_title('Validation ROC-AUC')

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print('✅ Training curves saved')

In [ ]:
# ── Cell 13: Save final model + metadata ─────────────────────────────────────

import json

# Load best weights
model.load_state_dict(torch.load(best_model_path))
model.eval()

# Save full checkpoint (weights + metadata)
final_path = '/content/purepick_skin_model.pt'
torch.save({
    'model_state_dict':  model.state_dict(),
    'conditions':        PUREPICK_CONDITIONS,
    'num_classes':       NUM_CLASSES,
    'image_size':        IMAGE_SIZE,
    'threshold':         THRESHOLD,
    'backbone':          'efficientnet_b4',
    'best_val_f1':       best_val_f1,
    'version':           '2.0',
}, final_path)

# Save class map as JSON (used by skin_classifier.py for label lookup)
class_map = {
    'conditions':   PUREPICK_CONDITIONS,
    'num_classes':  NUM_CLASSES,
    'image_size':   IMAGE_SIZE,
    'threshold':    THRESHOLD,
    'backbone':     'efficientnet_b4',
    'best_val_f1':  round(best_val_f1, 4),
    'version':      '2.0',
}
with open('/content/purepick_skin_classes.json', 'w') as f:
    json.dump(class_map, f, indent=2)

print(f'✅ Model saved: {final_path}')
print(f'   Size: {os.path.getsize(final_path) / 1e6:.1f} MB')
print(f'   Best F1: {best_val_f1:.4f}')
print(f'   Classes: {NUM_CLASSES}')

In [ ]:
# ── Cell 14: Download model files ────────────────────────────────────────────
# Downloads purepick_skin_model.pt and purepick_skin_classes.json
# Place both files in: backend/skin_analysis/ml_models/

from google.colab import files

print('Downloading model files...')
files.download('/content/purepick_skin_model.pt')
files.download('/content/purepick_skin_classes.json')
files.download('/content/training_curves.png')
print('✅ Download complete!')
print()
print('Next steps:')
print('  1. Move purepick_skin_model.pt     → backend/skin_analysis/ml_models/')
print('  2. Move purepick_skin_classes.json → backend/skin_analysis/ml_models/')
print('  3. Run: docker compose up -d --build')
print('  4. The skin engine will auto-detect and load the new model.')